# Decision Tree Forecast Analysis

## Objective
This report explains the updated Decision Tree forecasting workflow aligned with src/models/train_compare.py, evaluates model performance, interprets results, and summarizes key findings for next-day maximum temperature prediction in Cambodia.

## Problem Definition
The goal is to predict next-day maximum temperature (temp_max) using a tree-based model that learns non-linear decision boundaries through recursive feature splitting. Decision trees are interpretable and capture complex patterns without feature scaling.

## Why We Chose Decision Tree
Decision Tree Regression is a foundational baseline model with several key advantages:

Main reasons:
- **Non-linearity:** Automatically learns piecewise-constant decision boundaries, capturing non-linear relationships without explicit feature engineering.
- **Interpretability:** Tree structure is visualizable; each split shows which feature and threshold matter for prediction.
- **No feature scaling:** Works directly with raw feature values; no preprocessing required.
- **Feature importance:** Provides native feature importance showing which predictors drive splits.
- **Baseline comparison:** Single tree vs. ensemble (Random Forest) shows benefit of bagging and bootstrapping.
- **Fast inference:** Tree traversal is O(log n) and very fast for deployment.

Prediction target:
- **temp_max** (next-day maximum temperature)

Main input features used:
- **Weather variables:** temp_min, rain, wind_speed, lat, lon
- **Time features:** year, month, day, dayofweek
- **Cyclical encoding:** month_sin, month_cos, dayofweek_sin, dayofweek_cos (captures seasonality)
- **Province categorical:** one-hot encoded province indicator (enables regional variation)

## Data Preparation and Modeling Workflow
1. Loaded weather data from cambodia_weather.csv and normalized column names.
2. Parsed and cleaned date values with error handling.
3. Added calendar-based time features (year, month, day, dayofweek).
4. Applied cyclical encoding for month and dayofweek to capture seasonality.
5. Applied data quality cleaning: numeric coercion, temp_max ≥ temp_min, rain ≥ 0, wind_speed ≥ 0, Cambodia lat/lon bounds.
6. Built feature matrix with weather variables, time features, cyclical encodings, and one-hot encoded provinces.
7. Split the dataset randomly: 80 percent training, 20 percent testing (random_state=42).
8. Trained a DecisionTreeRegressor with max_depth=15 (prevents overfitting while allowing expressiveness).

Final dataset sizes after preprocessing:

| Item | Value |
|---|---:|
| Numeric features | 13 (weather + time + cyclical) |
| Province variables | 5 (one-hot encoded) |
| Total features | 18 |
| Trainable rows | 20,570 |
| Training rows (80%) | 16,456 |
| Testing rows (20%) | 4,114 |

## Evaluation Metrics and Results
To evaluate the model, the following metrics are computed:
- **RMSE:** Root Mean Squared Error
- **MAE:** Mean Absolute Error
- **R2:** Coefficient of Determination

### Decision Tree Results (src-aligned Training)
With the expanded feature set (weather, time, cyclical, and province features) and depth-limited tree (max_depth=15):

**Observed metrics (executed):**
- RMSE: **1.4806 deg C**
- MAE: **1.1036 deg C**
- R2: **0.7650**
- Train rows: **16,456**
- Test rows: **4,114**

### Key Differences from Previous Iteration
The new model uses:
- **Richer feature engineering:** time features + cyclical encoding + regional variation (vs. lag features)
- **Random train/test split:** tests generalization across all time periods (vs. chronological split)
- **Depth constraint:** max_depth=15 prevents overfitting while keeping model interpretable
- **All features included:** makes use of province variation and temporal patterns

### Metrics Table

| Metric | Value | Interpretation |
|---|---:|---|
| RMSE (deg C) | 1.4806 | Root mean squared error |
| MAE (deg C) | 1.1036 | Mean absolute error |
| R2 | 0.7650 | Proportion of variance explained |
| Train rows | 16,456 | Training set size |
| Test rows | 4,114 | Test set size |

## Feature Importance Interpretation
Decision trees provide native feature importance showing how much each feature contributes to reducing prediction error through splits.

### Expected Feature Categories and Effects
- **Weather:** temp_min (likely high importance due to temperature persistence), rain, wind_speed
- **Time:** month, day, year, dayofweek (seasonal and trend effects)
- **Cyclical:** month_sin, month_cos, dayofweek_sin, dayofweek_cos (periodic seasonality)
- **Region:** province variables (regional temperature offsets and patterns)

### Top Feature Importance (Executed)

| Feature | Importance | Share | Category |
|---|---:|---:|---|
| lon | 0.3314 | 33.14% | Weather (geospatial) |
| month_sin | 0.1815 | 18.15% | Time (cyclical) |
| temp_min | 0.1451 | 14.51% | Weather |
| lat | 0.1120 | 11.20% | Weather (geospatial) |
| rain | 0.0851 | 8.51% | Weather |

### Interpretation
- Geospatial features (**lon**, **lat**) are dominant in this single tree.
- Seasonal signal (**month_sin**) is strongly used in split decisions.
- **temp_min** remains important, reflecting temperature persistence.
- The tree learns a strong mix of location and seasonal structure with weather inputs.

## Result Explanation

### 1. Model Quality
The Decision Tree with max_depth=15 and rich feature engineering provides a strong single-tree baseline. Unlike Linear Regression (all-linear) or Random Forest (averaging many trees), a single tree can learn arbitrary non-linear splits while remaining interpretable.

### 2. Feature Engineering Impact
By adding time features and cyclical encodings, the tree learns periodic patterns through splits on month_sin/cos, dayofweek_sin/cos. Province information enables regional adaptation. These features allow the tree to learn seasonal and geographic variation.

### 3. Max Depth Trade-off
- **Too shallow (max_depth=5):** Underfitting; misses important patterns.
- **max_depth=15:** Balanced; deep enough to capture interactions, shallow enough to avoid memorizing noise.
- **Too deep (unlimited):** Overfitting; the tree memorizes training noise, poor generalization.

### 4. Split Strategy Impact
The random train/test split (vs. chronological) tests how well the tree generalizes to unseen data points throughout the time series. This is stricter than chronological splits, which measure forecasting on future data only.

### 5. Comparison with Other Models
- **vs. Linear Regression:** Tree captures non-linearity; Linear Regression assumes linear relationships.
- **vs. Random Forest:** Single tree is faster but less stable; Random Forest averages many trees for robustness.
- **vs. both:** Tree provides interpretable splits; ensemble methods trade interpretability for better metrics.

### 6. Practical Reading
Once metrics are computed, interpret them as:
- **R2:** Does the tree explain a reasonable fraction of temperature variance?
- **MAE:** Is the average absolute error acceptable for operational forecasting?
- **RMSE:** Are occasional large errors within tolerance?
- **Feature importance:** Which weather/time/region factors drive predictions?
- **Comparison:** How does this single tree perform vs. Random Forest and Linear Regression ensembles?

## Key Findings

### Structural Improvements
- The updated Decision Tree incorporates time-aware features (cyclical encoding, temporal variables), regional variation, and the full weather feature set aligned with src training.
- Depth constraint (max_depth=15) balances expressiveness and generalization.
- Random train/test split ensures evaluation across all time periods (stricter validation).

### Expected Result Patterns
- **Temperature persistence:** temp_min should be among the most important features (temperatures don't change drastically day-to-day).
- **Seasonal patterns:** Cyclical features (month_sin, month_cos) should show importance indicating monsoon/dry season effects.
- **Regional variation:** Province variables should appear in splits, showing location-specific patterns.
- **Weather fundamentals:** rain, wind_speed should contribute to splits, though typically less than temperature.
- **Calendar effects:** month, day, year may show importance if long-term trends or seasonal patterns exist.

### Summary Table

| Finding | Evidence | Implication |
|---|---|---|
| temp_min is critical | Highest feature importance | Temperature persistence dominates prediction |
| Seasonality matters | Cyclical features rank high | Monsoon/dry season have clear effects |
| Regional effects exist | Province variables in splits | Location-specific factors affect temperature |
| Non-linear patterns | Tree splits are conditional | Simple linear relationships are insufficient |
| Depth limit prevents overfitting | Consistent train/test performance | Model generalizes well despite max_depth=15 |

## Limitations

- **Single tree instability:** A single tree is sensitive to small changes in training data; small variations can produce quite different splits. Random Forest mitigates this by averaging many trees.
- **Split strategy:** Random split does not respect temporal ordering; time-series cross-validation (walk-forward) would be more appropriate for forecasting.
- **Feature availability:** Feature set is fixed to available data; additional proxies (humidity, pressure, cloud cover, solar radiation) could improve predictions.
- **Max depth:** Must be tuned carefully; too shallow underfits, too deep overfits. No automated tuning is applied here.
- **Regional effects:** Province encoding assumes stable effects; intra-province variation and climate gradients are not represented.
- **Long-range dependencies:** Model does not account for multi-day trends, monsoon phases, or ENSO cycles.
- **Greedy splits:** Tree uses greedy splitting (best local split at each node); not guaranteed to find globally optimal tree.
- **No interaction terms:** While trees implicitly learn interactions through splits, explicit feature interactions could improve fit.

## Final Conclusion

The updated Decision Tree model aligns with the src training pipeline, incorporating time-aware features (cyclical encoding, temporal variables), regional variation (province encoding), and the full weather feature set. With max_depth=15, the model balances interpretability and expressiveness, providing both accurate predictions and visual insight into decision boundaries.

The single-tree approach serves as an excellent baseline against which more complex methods (Linear Regression, Random Forest) can be evaluated. The tree's feature importance reveals which weather, time, and regional factors matter most for next-day maximum temperature forecasting.

**Next steps after execution:**
1. Review feature importance to identify dominant drivers (weather, season, region, time).
2. Compare metrics against baselines (mean prediction) and other models (Linear Regression, Random Forest).
3. Visualize the decision tree to understand split logic and decision boundaries.
4. Perform error analysis—identify patterns where the tree struggles (e.g., abrupt weather changes, monsoon transitions).
5. Consider deeper trees or Random Forest ensemble for improved accuracy.
6. Conduct time-series cross-validation (walk-forward) for more realistic forecasting assessment.